# Завдання
Розробити програмний скрипт, що реалізує аналіз даних, поданих у файлі Data_Set_1.xlsx.
Розробити програмний скрипт, що реалізує аналіз даних, самостійно обраних процесів.
Обов’язковою вимогою є аналіз множини процесів, поданих часовими рядами із різними
властивостями.

# Imports

In [ ]:
from typing import List

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from pandas import DataFrame
import seaborn as sns
import warnings

warnings.filterwarnings("ignore")

# Load data

In [ ]:
df1 = pd.read_excel("data/Data_Set_1_lr8.xlsx", sheet_name="Group PC+LCV ", header=[5, 7])

# filling NaNs in second level to ensure there is a data
df1.columns = pd.MultiIndex.from_tuples([
    (top, bottom if str(bottom) != 'nan' else top)
    for top, bottom in df1.columns
])

# flattening the MultiIndex for simpler analysis
df1.columns = [f'{top}_{bottom}' for top, bottom in df1.columns]

# cleaning underscores
df1.columns = [col.replace('__', '_') for col in df1.columns]

# Removing header
df1 = df1[df1.iloc[:, 0].notna()]
df1.reset_index(drop=True, inplace=True)

# Drop the second column
df1 = df1.drop(df1.columns[1], axis=1)

# renaming the first column
df1 = df1.rename(columns={df1.columns[0]: 'Region_Group'})
# Preview the cleaned dataframe
print(df1.head())
print(df1.columns)


# second

In [ ]:
filepath = "data/Data_Set_1_lr8.xlsx"


def read_section(filepath, skiprows=0, nrows=None, named_columns=List[str]) -> DataFrame:
    if nrows is not None:
        df = pd.read_excel(filepath, sheet_name="Sales by Model", header=None,
                           skiprows=skiprows, nrows=nrows)
    else:
        df = pd.read_excel(filepath, sheet_name="Sales by Model", header=None,
                           skiprows=skiprows)

    # malformed fixed columns + named columns
    df.columns = ["col1", "Type", "Brand", "Model", "col5", "Total", "col6"] + named_columns

    # dropping helper columns
    df = df.drop(columns=["col1", "col5", "col6"])

    # F-fill merged cells
    for col in ["Type", "Brand", "Model"]:
        if col in df.columns:
            df[col] = df[col].ffill()

    # detecting TOTAL rows
    df['IsTotal'] = df['Total'].astype(str).str.contains('TOTAL', na=False)

    # replace Model and keep Brand
    df.loc[df['IsTotal'], 'Model'] = "TOTAL"
    df.loc[df['IsTotal'], 'BrandTotal'] = df['Brand']

    # converting numeric columns
    numeric_cols = named_columns
    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0).astype(int)

    # computing total from numeric columns
    df.loc[df['IsTotal'], 'Total'] = df.loc[df['IsTotal'], numeric_cols].sum(axis=1)

    # For non-TOTAL rows, parsing the "Total" column
    df.loc[~df['IsTotal'], 'Total'] = (
        df.loc[~df['IsTotal'], 'Total']
        .astype(str)
        .str.replace(r"[^\d\-]", "", regex=True)
        .replace("", "0").astype(int)
    )

    # helper column to sum-up
    df['Level'] = df['IsTotal'].map({True: "TOTAL", False: "INTERPERIOD"})

    df.drop(columns=['IsTotal'], inplace=True)

    # converting numeric columns to integers
    numeric_cols = named_columns
    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0).astype(int)

    df = df.reset_index(drop=True)

    return df


# Read Europe Region
df_europe = read_section(
    filepath, skiprows=5, nrows=98,
    named_columns=["2019_Dec", "2019_YTD", "2020_Dec", "2020_YTD"]
)

# Read Worldwide
df_worldwide = read_section(
    filepath, skiprows=103,
    named_columns=["2017_Dec", "2017_YTD", "2018_Dec", "2018_YTD"]
)

# Union columns
all_cols = ["Type", "Brand", "Model", "Level", "Total", "BrandTotal",
            "2017_Dec", "2017_YTD", "2018_Dec", "2018_YTD",
            "2019_Dec", "2019_YTD", "2020_Dec", "2020_YTD"]

# check for columns before concat
for col in all_cols:
    if col not in df_europe.columns:
        df_europe[col] = 0
    if col not in df_worldwide.columns:
        df_worldwide[col] = 0

# Reorder
df_europe = df_europe[all_cols]
df_worldwide = df_worldwide[all_cols]

df_europe['Region'] = 'EUROPE'
df_worldwide['Region'] = 'Worldwide'
df_concat = pd.concat([df_europe, df_worldwide], ignore_index=True)

print(df_concat.head(20))


# Brand-Level Summary

In [ ]:
brand_summary = (
    df_concat.groupby(['Region', 'Brand'])
    .agg({
        'Total': 'sum',
        '2017_Dec': 'sum', '2017_YTD': 'sum',
        '2018_Dec': 'sum', '2018_YTD': 'sum',
        '2019_Dec': 'sum', '2019_YTD': 'sum',
        '2020_Dec': 'sum', '2020_YTD': 'sum'
    })
    .sort_values(['Region', 'Total'], ascending=[True, False])
)

print(brand_summary.head(20))

# Top Models by Region

In [ ]:
top_models = (
    df_concat[df_concat['Model'] != "TOTAL"]
    .groupby(['Region', 'Brand', 'Model'])
    .agg({
        '2019_YTD': 'sum',
        '2020_YTD': 'sum',
        '2017_YTD': 'sum',
        '2018_YTD': 'sum',
    })
    .sort_values('2020_YTD', ascending=False)
)

print(top_models.head(20))


# PC vs LCV

In [ ]:
type_summary = (
    df_concat.groupby(['Region', 'Type'])
    .agg({
        'Total': 'sum',
        '2020_YTD': 'sum',
        '2019_YTD': 'sum'
    })
)

print(type_summary)


# Year-over-Year Growth

In [ ]:
growth = df_concat.groupby(['Region']).agg({
    '2017_YTD': 'sum',
    '2018_YTD': 'sum',
    '2019_YTD': 'sum',
    '2020_YTD': 'sum'
})

growth['YoY_17_18'] = (growth['2018_YTD'] - growth['2017_YTD']) / growth['2017_YTD'] * 100
growth['YoY_18_19'] = (growth['2019_YTD'] - growth['2018_YTD']) / growth['2018_YTD'] * 100
growth['YoY_19_20'] = (growth['2020_YTD'] - growth['2019_YTD']) / growth['2019_YTD'] * 100

print(growth)


# Pareto (80/20) Brand Concentration

In [ ]:
brand_totals = (
    df_concat[df_concat['Model'] == "TOTAL"]
    .groupby('Brand')
    ['Total'].sum()
    .sort_values(ascending=False)
)

brand_totals_cum = brand_totals.cumsum() / brand_totals.sum() * 100
print(pd.DataFrame({'Total': brand_totals, 'Cum%': brand_totals_cum}))


# Time Series Visualization

In [ ]:
df_eu_totals = df_europe.groupby('Brand')[['2019_Dec', '2019_YTD', '2020_Dec', '2020_YTD']].sum()
df_eu_totals.plot(kind='bar', figsize=(12, 6))
plt.show()

# Comparative analysis

In [ ]:
# compare brand-market shares regionally
def regional_share_table(region):
    d = df_concat[df_concat['Region'] == region]
    if 'aggregate_total' not in d.columns:
        d['aggregate_total'] = d[[c for c in d.columns if c.endswith('YTD') or c.endswith('Dec')]].sum(axis=1)
    s = d.groupby('Brand')['aggregate_total'].sum().sort_values(ascending=False)
    return s.reset_index().rename(columns={'aggregate_total': 'Total'})


# compute growth rates per brand between two columns
def brand_growth(col_a, col_b, region=None):
    d = df_concat.copy()
    if region: d = d[d['Region'] == region]
    grp = d.groupby('Brand')[[col_a, col_b]].sum().reset_index()
    grp['growth_pct'] = (grp[col_b] - grp[col_a]) / grp[col_a].replace({0: np.nan}) * 100
    return grp.sort_values('growth_pct', ascending=False)


# top N models delta between two periods
def top_model_delta(period_a, period_b, region='Europe', top_n=20):
    d = df_concat[df_concat['Region'] == region]
    m = d.groupby(['Brand', 'Model'])[[period_a, period_b]].sum().reset_index()
    m['delta'] = m[period_b] - m[period_a]
    return m.sort_values('delta', ascending=False).head(top_n)


# Basic stats
print("Rows:", df_concat.shape[0])
print("Brands:", df_concat['Brand'].nunique())
print("Models:", df_concat['Model'].nunique())

# Growth rates per brand between two columns
brand_growth_eu = brand_growth("2019_YTD", "2020_YTD", region="Europe")
print(brand_growth_eu)
brand_growth_ww = brand_growth("2017_YTD", "2018_YTD", region="Worldwide")
print(brand_growth_ww)

# Regional brand-market-share table
europe_share = regional_share_table("Europe")
world_share = regional_share_table("Worldwide")

print("Europe Brand Market Share:")
print(europe_share)

print("\nWorldwide Brand Market Share:")
print(world_share)

# top 20 models in time delta
models_delta = top_model_delta("2019_Dec", "2020_Dec",
                               region="Europe", top_n=20)
print(models_delta)

# Missing data
missing = df_concat.isna().sum()
print("Missing values per column:\n", missing[missing > 0])

# Distribution of totals
df_concat['aggregate_total'] = df_concat[
    [c for c in df_concat.columns if c.endswith("YTD") or c.endswith("Dec")]
].sum(axis=1)

plt.figure(figsize=(12, 6))
sns.histplot(df_concat['aggregate_total'].replace(0, np.nan).dropna(), bins=50, log_scale=(False, True))
plt.title("Distribution of aggregate_total (log-scaled y-axis)")
plt.show()
